# Jia Program Scoring On DIV30

Standalone Slurm-executed notebook for scoring the Jia et al. 2026 Science RGC/IPC programs on the converted DIV30 AnnData object. This notebook imports reusable methods from `mge_organoid_python.gene_program_scoring` and writes run outputs under `PROJECT_ROOT/results/jia_program_div30_scoring/<run_label>/`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "python_notebooks" / "src").exists() and (candidate / "slurm_templates").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_ROOT = REPO_ROOT / "python_notebooks" / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from mge_organoid_python.loader import cached_h5ad_path
from mge_organoid_python.paths import resolve_project_root
from mge_organoid_python.studies import default_studies
from mge_organoid_python.gene_program_scoring import (
    attach_resolution_assignments,
    best_matches,
    choose_first_existing,
    choose_umap_key,
    match_program_genes,
    parse_csv_list,
    parse_optional_float,
    parse_optional_int,
    plot_program_proportion_dotplot,
    plot_score_heatmap,
    plot_umap_score_overlay_panel,
    programs_from_marker_table,
    read_marker_program_csv,
    score_output_obs_table,
    score_programs_scanpy,
    score_threshold_table,
    select_program_markers,
    summarize_scores_by_group,
)

In [ ]:
def env_bool(name, default=False):
    raw = os.environ.get(name)
    if raw is None:
        return bool(default)
    return raw.strip().lower() in {"1", "true", "yes", "y"}

project_root = resolve_project_root(os.environ.get("PROJECT_ROOT"))
results_dirname = os.environ.get("JIA_PROGRAM_RESULTS_DIRNAME", "jia_program_div30_scoring")
run_label = os.environ.get("JIA_PROGRAM_RUN_LABEL", "jia_program_div30_scoring_v1")
study_id = os.environ.get("JIA_PROGRAM_STUDY_ID", "varela_div30")

marker_csv = Path(
    os.environ.get(
        "JIA_PROGRAM_CSV",
        str(project_root / "reference" / "Jia_et_al_2026_Science_3_progs.csv"),
    )
)

studies = {study.study_id: study for study in default_studies()}
if study_id not in studies:
    raise ValueError(f"Unknown study_id {study_id!r}. Valid values: {sorted(studies)}")
study = studies[study_id]

h5ad_path = Path(os.environ.get("JIA_PROGRAM_H5AD") or cached_h5ad_path(study, project_root=project_root))
resolution_assignments = Path(
    os.environ.get(
        "JIA_PROGRAM_RESOLUTION_ASSIGNMENTS",
        str(
            project_root
            / "results"
            / "seurat_cluster_resolution_sweep"
            / "seurat_cluster_resolution_sweep_v1"
            / study_id
            / "tables"
            / "resolution_sweep_cluster_assignments_wide.tsv"
        ),
    )
)

program_order = parse_csv_list(os.environ.get("JIA_PROGRAM_ORDER", "IPC,RGC1,RGC2"))
top_n = parse_optional_int(os.environ.get("JIA_PROGRAM_TOP_N"))
min_avg_log2fc = parse_optional_float(os.environ.get("JIA_PROGRAM_MIN_AVG_LOG2FC"))
max_p_val_adj = parse_optional_float(os.environ.get("JIA_PROGRAM_MAX_P_VAL_ADJ"))
ctrl_size = int(os.environ.get("JIA_PROGRAM_CTRL_SIZE", "50"))
random_state = int(os.environ.get("JIA_PROGRAM_RANDOM_STATE", "0"))
high_score_quantile = float(os.environ.get("JIA_PROGRAM_HIGH_SCORE_QUANTILE", "0.9"))
base_cluster_override = os.environ.get("JIA_PROGRAM_BASE_CLUSTER_COL", "").strip() or None
save_plots = env_bool("JIA_PROGRAM_SAVE_PLOTS", True)
show_plots = env_bool("JIA_PROGRAM_SHOW_PLOTS", False)

run_dir = project_root / "results" / results_dirname / run_label
table_dir = run_dir / "tables"
plot_dir = run_dir / "plots"
log_dir = run_dir / "logs"
for directory in [table_dir, plot_dir, log_dir]:
    directory.mkdir(parents=True, exist_ok=True)

params = pd.DataFrame(
    [
        {"parameter": "project_root", "value": str(project_root)},
        {"parameter": "repo_root", "value": str(REPO_ROOT)},
        {"parameter": "results_dirname", "value": results_dirname},
        {"parameter": "run_label", "value": run_label},
        {"parameter": "study_id", "value": study_id},
        {"parameter": "marker_csv", "value": str(marker_csv)},
        {"parameter": "h5ad_path", "value": str(h5ad_path)},
        {"parameter": "resolution_assignments", "value": str(resolution_assignments)},
        {"parameter": "program_order", "value": ",".join(program_order)},
        {"parameter": "top_n_per_program", "value": "" if top_n is None else str(top_n)},
        {"parameter": "min_avg_log2fc", "value": "" if min_avg_log2fc is None else str(min_avg_log2fc)},
        {"parameter": "max_p_val_adj", "value": "" if max_p_val_adj is None else str(max_p_val_adj)},
        {"parameter": "scanpy_score_genes_ctrl_size", "value": str(ctrl_size)},
        {"parameter": "random_state", "value": str(random_state)},
        {"parameter": "high_score_quantile", "value": str(high_score_quantile)},
        {"parameter": "save_plots", "value": str(save_plots)},
        {"parameter": "show_plots", "value": str(show_plots)},
    ]
)
params.to_csv(table_dir / "jia_program_run_parameters.tsv", sep="	", index=False)

print("[JiaProgramScoring] repo_root", REPO_ROOT, flush=True)
print("[JiaProgramScoring] project_root", project_root, flush=True)
print("[JiaProgramScoring] run_dir", run_dir, flush=True)
print("[JiaProgramScoring] marker_csv", marker_csv, flush=True)
print("[JiaProgramScoring] h5ad_path", h5ad_path, flush=True)
print("[JiaProgramScoring] resolution_assignments", resolution_assignments, flush=True)

In [ ]:
markers = read_marker_program_csv(marker_csv, gene_col="gene", program_col="cluster")
selected_markers = select_program_markers(
    markers,
    gene_col="gene",
    program_col="cluster",
    top_n=top_n,
    min_avg_log2fc=min_avg_log2fc,
    max_p_val_adj=max_p_val_adj,
    sort_by="avg_log2FC",
)
programs = programs_from_marker_table(
    selected_markers,
    gene_col="gene",
    program_col="cluster",
    program_order=program_order,
)

markers.to_csv(table_dir / "jia_program_markers_full.tsv", sep="	", index=False)
selected_markers.to_csv(table_dir / "jia_program_markers_selected.tsv", sep="	", index=False)
program_summary = pd.DataFrame(
    [
        {"program": program, "n_selected_marker_genes": len(genes)}
        for program, genes in programs.items()
    ]
)
program_summary.to_csv(table_dir / "jia_program_marker_selection_summary.tsv", sep="	", index=False)

print("[JiaProgramScoring] marker rows", len(markers), flush=True)
print("[JiaProgramScoring] selected marker rows", len(selected_markers), flush=True)
print(program_summary.to_string(index=False), flush=True)

In [ ]:
if not h5ad_path.exists():
    raise FileNotFoundError(f"Missing DIV30 AnnData input: {h5ad_path}")

print("[JiaProgramScoring] loading AnnData in memory on Slurm", flush=True)
adata = sc.read_h5ad(h5ad_path)
print(f"[JiaProgramScoring] loaded shape={adata.n_obs} cells x {adata.n_vars} genes", flush=True)
print(f"[JiaProgramScoring] obsm keys={list(adata.obsm.keys())}", flush=True)

base_cluster_col = base_cluster_override or choose_first_existing(
    adata.obs.columns,
    ["seurat_clusters", "RNA_snn_res.0.2"],
)
if base_cluster_col is None:
    raise KeyError("Could not find seurat_clusters or RNA_snn_res.0.2 in adata.obs")
adata.obs[base_cluster_col] = adata.obs[base_cluster_col].astype("category")
umap_key = choose_umap_key(adata)
resolution_cols = attach_resolution_assignments(adata, resolution_assignments)

matched_programs, overlap_summary, overlap_detail = match_program_genes(programs, adata.var_names)
overlap_summary.to_csv(table_dir / "jia_program_gene_overlap_summary.tsv", sep="	", index=False)
overlap_detail.to_csv(table_dir / "jia_program_gene_overlap_detail.tsv", sep="	", index=False)

print("[JiaProgramScoring] base_cluster_col", base_cluster_col, flush=True)
print("[JiaProgramScoring] umap_key", umap_key, flush=True)
print("[JiaProgramScoring] resolution columns", len(resolution_cols), flush=True)
print(overlap_summary.to_string(index=False), flush=True)

In [ ]:
score_columns = score_programs_scanpy(
    adata,
    matched_programs,
    score_prefix="jia_score_",
    ctrl_size=ctrl_size,
    random_state=random_state,
)
thresholds = score_threshold_table(adata.obs, score_columns, quantile=high_score_quantile)
thresholds.to_csv(table_dir / "jia_program_score_thresholds.tsv", sep="	", index=False)

extra_columns = [base_cluster_col, "RNA_snn_res.0.2", "orig.ident", "sample"] + resolution_cols
obs_scores = score_output_obs_table(adata, score_columns, extra_columns=extra_columns)
obs_scores.to_csv(table_dir / "div30_jia_program_scores_obs.tsv", sep="	", index=False)

base_summary = summarize_scores_by_group(adata.obs, score_columns, base_cluster_col, thresholds=thresholds)
base_summary.to_csv(table_dir / "div30_jia_program_summary_by_seurat_clusters.tsv", sep="	", index=False)

sweep_summaries = []
for col in resolution_cols:
    sweep_summaries.append(summarize_scores_by_group(adata.obs, score_columns, col, thresholds=thresholds))
if sweep_summaries:
    sweep_summary = pd.concat(sweep_summaries, ignore_index=True)
else:
    sweep_summary = pd.DataFrame()
sweep_summary.to_csv(table_dir / "div30_jia_program_summary_by_resolution_sweep.tsv", sep="	", index=False)

all_summary = pd.concat([base_summary, sweep_summary], ignore_index=True)
all_summary.to_csv(table_dir / "div30_jia_program_summary_all_groupings.tsv", sep="	", index=False)
best = best_matches(all_summary, top_n=5)
best.to_csv(table_dir / "div30_jia_program_best_matches_by_grouping.tsv", sep="	", index=False)

print("[JiaProgramScoring] score columns", score_columns, flush=True)
print("[JiaProgramScoring] wrote per-cell score table", len(obs_scores), flush=True)
print("[JiaProgramScoring] base summary rows", len(base_summary), flush=True)
print("[JiaProgramScoring] sweep summary rows", len(sweep_summary), flush=True)

In [ ]:
plot_paths = []
if save_plots:
    panel_path = plot_dir / "div30_umap_seurat_clusters_jia_program_scores_panel.png"
    fig = plot_umap_score_overlay_panel(
        adata,
        score_columns,
        output_path=panel_path,
        base_color_col=base_cluster_col,
        umap_key=umap_key,
        title_prefix="DIV30",
    )
    plot_paths.append(panel_path)
    if show_plots:
        display(fig)
    else:
        plt.close(fig)

    heatmap_mean_path = plot_dir / "div30_jia_program_mean_score_by_seurat_clusters_heatmap.png"
    fig = plot_score_heatmap(
        base_summary,
        output_path=heatmap_mean_path,
        value_col="mean_score",
        title="DIV30 mean Jia program score by Seurat cluster",
    )
    plot_paths.append(heatmap_mean_path)
    if show_plots:
        display(fig)
    else:
        plt.close(fig)

    heatmap_fraction_path = plot_dir / "div30_jia_program_fraction_high_by_seurat_clusters_heatmap.png"
    fig = plot_score_heatmap(
        base_summary,
        output_path=heatmap_fraction_path,
        value_col="fraction_high_score",
        title="DIV30 fraction high Jia program score by Seurat cluster",
    )
    plot_paths.append(heatmap_fraction_path)
    if show_plots:
        display(fig)
    else:
        plt.close(fig)

    dotplot_path = plot_dir / "div30_jia_program_high_score_proportion_dotplot_by_seurat_clusters.png"
    fig = plot_program_proportion_dotplot(
        base_summary,
        output_path=dotplot_path,
        proportion_col="fraction_high_score",
        color_col="mean_score",
        title="DIV30 Jia high-score proportion by Seurat cluster",
    )
    plot_paths.append(dotplot_path)
    if show_plots:
        display(fig)
    else:
        plt.close(fig)

print("[JiaProgramScoring] plot files")
for path in plot_paths:
    print(path, flush=True)

In [ ]:
manifest_records = []
for kind, directory in [("table", table_dir), ("plot", plot_dir), ("log", log_dir)]:
    for path in sorted(directory.glob("*")):
        if path.is_file():
            manifest_records.append(
                {
                    "kind": kind,
                    "path": str(path),
                    "bytes": path.stat().st_size,
                }
            )
manifest = pd.DataFrame(manifest_records)
manifest.to_csv(table_dir / "jia_program_output_manifest.tsv", sep="	", index=False)

complete = pd.DataFrame(
    [
        {
            "status": "complete",
            "run_label": run_label,
            "study_id": study_id,
            "n_cells": int(adata.n_obs),
            "n_genes": int(adata.n_vars),
            "base_cluster_col": base_cluster_col,
            "umap_key": umap_key,
            "n_programs_scored": len(score_columns),
            "n_resolution_groupings": len(resolution_cols),
            "run_dir": str(run_dir),
        }
    ]
)
complete.to_csv(table_dir / "jia_program_div30_scoring_complete.tsv", sep="	", index=False)
print(
    f"[JiaProgramScoring] complete run_label={run_label} run_dir={run_dir} programs={len(score_columns)}",
    flush=True,
)